In [1]:
!pip install streamlit pandas pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 102.6 MB/s eta 0:00:00


In [3]:
%%writefile konfigurasi.py

import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

NAMA_DB = 'pengeluaran_harian.db'

DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

KATEGORI_PENGELUARAN = [
    "Makanan",
    "Transportasi",
    "Hiburan",
    "Tagihan",
    "Belanja",
    "Kesehatan",
    "Pendidikan",
    "Lainnya"
]

KATEGORI_DEFAULT = "Lainnya"

Overwriting konfigurasi.py


In [4]:
%%writefile setup_db_pengeluaran.py

import sqlite3
import os

from konfigurasi import DB_PATH


def setup_database():

    print(f"Memeriksa/membuat database di: {DB_PATH}")

    conn = None

    try:
        conn = sqlite3.connect(DB_PATH)

        cursor = conn.cursor()

        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );
        """

        print("Membuat tabel 'transaksi'...")

        cursor.execute(sql_create_table)

        conn.commit()

        print("-> Tabel transaksi siap.")

        return True

    except sqlite3.Error as e:
        print(f"Error SQLite: {e}")

        return False

    finally:
        if conn:
            conn.close()

            print("-> Koneksi database ditutup.")


if __name__ == "__main__":

    print("--- Setup Database ---")

    if setup_database():
        print(
            f"\nSetup database "
            f"'{os.path.basename(DB_PATH)}' selesai."
        )

    else:
        print("\nSetup database GAGAL.")

    print("--- Setup Selesai ---")

Writing setup_db_pengeluaran.py


In [5]:
!python setup_db_pengeluaran.py

--- Setup Database ---
Memeriksa/membuat database di: /content/pengeluaran_harian.db
Membuat tabel 'transaksi'...
-> Tabel transaksi siap.
-> Koneksi database ditutup.

Setup database 'pengeluaran_harian.db' selesai.
--- Setup Selesai ---


In [6]:
%%writefile database.py

import sqlite3
import pandas as pd

from konfigurasi import DB_PATH


def get_db_connection():

    try:
        conn = sqlite3.connect(
            DB_PATH,
            timeout=10,
            detect_types=sqlite3.PARSE_DECLTYPES
        )

        conn.row_factory = sqlite3.Row

        return conn

    except sqlite3.Error as e:
        print(f"ERROR koneksi database: {e}")

        return None


def execute_query(query, params=None):

    conn = get_db_connection()

    if not conn:
        return None

    last_id = None

    try:
        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)

        else:
            cursor.execute(query)

        conn.commit()

        last_id = cursor.lastrowid

        return last_id

    except sqlite3.Error as e:
        print(f"ERROR query: {e}")

        conn.rollback()

        return None

    finally:
        if conn:
            conn.close()


def fetch_query(query, params=None, fetch_all=True):

    conn = get_db_connection()

    if not conn:
        return None

    try:
        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)

        else:
            cursor.execute(query)

        result = (
            cursor.fetchall()
            if fetch_all
            else cursor.fetchone()
        )

        return result

    except sqlite3.Error as e:
        print(f"ERROR fetch: {e}")

        return None

    finally:
        if conn:
            conn.close()


def get_dataframe(query, params=None):

    conn = get_db_connection()

    if not conn:
        return pd.DataFrame()

    try:
        df = pd.read_sql_query(
            query,
            conn,
            params=params
        )

        return df

    except Exception as e:
        print(f"Gagal membaca dataframe: {e}")

        return pd.DataFrame()

    finally:
        if conn:
            conn.close()


def setup_database_initial():

    print(f"Memeriksa database: {DB_PATH}")

    conn = get_db_connection()

    if not conn:
        return False

    try:
        cursor = conn.cursor()

        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            deskripsi TEXT NOT NULL,
            jumlah REAL NOT NULL CHECK(jumlah > 0),
            kategori TEXT,
            tanggal DATE NOT NULL
        );
        """

        cursor.execute(sql_create_table)

        conn.commit()

        print("-> Tabel transaksi siap.")

        return True

    except sqlite3.Error as e:
        print(f"Error setup tabel: {e}")

        return False

    finally:
        if conn:
            conn.close()

Writing database.py


In [7]:
%%writefile model.py

import datetime


class Transaksi:

    def __init__(
        self,
        deskripsi,
        jumlah,
        kategori,
        tanggal,
        id_transaksi=None
    ):

        self.id = id_transaksi

        self.deskripsi = (
            str(deskripsi)
            if deskripsi
            else "Tanpa Deskripsi"
        )

        try:
            jumlah_float = float(jumlah)

            self.jumlah = (
                jumlah_float
                if jumlah_float > 0
                else 0.0
            )

        except (ValueError, TypeError):
            self.jumlah = 0.0

        self.kategori = (
            str(kategori)
            if kategori
            else "Lainnya"
        )

        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal

        elif isinstance(tanggal, str):

            try:
                self.tanggal = datetime.datetime.strptime(
                    tanggal,
                    "%Y-%m-%d"
                ).date()

            except ValueError:
                self.tanggal = datetime.date.today()

        else:
            self.tanggal = datetime.date.today()

    def __repr__(self):

        return (
            f"Transaksi("
            f"ID={self.id}, "
            f"Deskripsi={self.deskripsi}, "
            f"Jumlah={self.jumlah}, "
            f"Kategori={self.kategori}, "
            f"Tanggal={self.tanggal}"
            f")"
        )

    def to_dict(self):

        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

Writing model.py


In [8]:
%%writefile manajer_anggaran.py

import datetime
import pandas as pd

from model import Transaksi

import database


class AnggaranHarian:

    _db_setup_done = False

    def __init__(self):

        if not AnggaranHarian._db_setup_done:

            print("Cek database awal...")

            if database.setup_database_initial():

                AnggaranHarian._db_setup_done = True

                print("Database siap.")

            else:
                print("Setup database gagal.")

    def tambah_transaksi(self, transaksi):

        if (
            not isinstance(transaksi, Transaksi)
            or transaksi.jumlah <= 0
        ):
            return False

        sql = """
        INSERT INTO transaksi
        (deskripsi, jumlah, kategori, tanggal)
        VALUES (?, ?, ?, ?)
        """

        params = (
            transaksi.deskripsi,
            transaksi.jumlah,
            transaksi.kategori,
            transaksi.tanggal.strftime("%Y-%m-%d")
        )

        last_id = database.execute_query(sql, params)

        if last_id is not None:

            transaksi.id = last_id

            return True

        return False

    def get_dataframe_transaksi(self):

        query = """
        SELECT
            id,
            tanggal,
            kategori,
            deskripsi,
            jumlah
        FROM transaksi
        ORDER BY tanggal DESC
        """

        return database.get_dataframe(query)

    def hitung_total_pengeluaran(self):

        sql = "SELECT SUM(jumlah) FROM transaksi"

        result = database.fetch_query(
            sql,
            fetch_all=False
        )

        if result and result[0] is not None:
            return float(result[0])

        return 0.0


    # ==========================
    # PENUGASAN
    # ==========================

    def hapus_transaksi(self, id_transaksi):

        sql = "DELETE FROM transaksi WHERE id = ?"

        result = database.execute_query(
            sql,
            (id_transaksi,)
        )

        if result is not None:
            return True

        return False

Writing manajer_anggaran.py


In [9]:
%%writefile streamlit_app.py

import streamlit as st
import datetime
import pandas as pd

from model import Transaksi
from manajer_anggaran import AnggaranHarian
from konfigurasi import KATEGORI_PENGELUARAN


st.set_page_config(
    page_title="Catatan Pengeluaran",
    layout="wide"
)


@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()


anggaran = get_anggaran_manager()


def format_rp(angka):
    return f"Rp {angka:,.0f}".replace(",", ".")


st.title("Aplikasi Pengeluaran Harian")


menu = st.sidebar.radio(
    "Pilih Menu",
    ["Tambah", "Riwayat", "Ringkasan"]
)


# =================================
# MENU TAMBAH
# =================================

if menu == "Tambah":

    st.header("Tambah Pengeluaran")

    with st.form("form_tambah"):

        deskripsi = st.text_input("Deskripsi")

        kategori = st.selectbox(
            "Kategori",
            KATEGORI_PENGELUARAN
        )

        jumlah = st.number_input(
            "Jumlah",
            min_value=1.0,
            step=1000.0
        )

        tanggal = st.date_input(
            "Tanggal",
            value=datetime.date.today()
        )

        submit = st.form_submit_button("Simpan")

        if submit:

            transaksi = Transaksi(
                deskripsi,
                jumlah,
                kategori,
                tanggal
            )

            if anggaran.tambah_transaksi(transaksi):
                st.success("Transaksi berhasil disimpan")

            else:
                st.error("Gagal menyimpan transaksi")


# =================================
# MENU RIWAYAT
# =================================

elif menu == "Riwayat":

    st.header("Riwayat Transaksi")

    df = anggaran.get_dataframe_transaksi()

    st.dataframe(
        df,
        use_container_width=True
    )

    st.subheader("Hapus Transaksi")

    id_hapus = st.number_input(
        "Masukkan ID transaksi",
        min_value=1,
        step=1
    )

    if st.button("Hapus Transaksi"):

        if anggaran.hapus_transaksi(id_hapus):

            st.success("Transaksi berhasil dihapus")

            st.rerun()

        else:
            st.error("Gagal menghapus transaksi")


# MENU RINGKASAN

elif menu == "Ringkasan":

    st.header("Ringkasan Pengeluaran")

    total = anggaran.hitung_total_pengeluaran()

    st.metric(
        "Total Pengeluaran",
        format_rp(total)
    )

    df = anggaran.get_dataframe_transaksi()

    if not df.empty:

        kategori = (
            df.groupby("kategori")["jumlah"]
            .sum()
        )

        st.bar_chart(kategori)

Writing streamlit_app.py


In [10]:
from pyngrok import ngrok

In [11]:
!streamlit run streamlit_app.py &>/content/logs.txt &

In [12]:
public_url = ngrok.connect(8501)

print(public_url)

ERROR:pyngrok.process.ngrok:t=2026-05-26T13:57:24+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-26T13:57:24+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-05-26T13:57:24+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [15]:
ngrok.set_auth_token("3EGTcrDAR8SOMoO07KQYieQHjuV_2NtFFTiev7A3CKe3sBZ4T")

In [16]:
!streamlit run streamlit_app.py &>/content/logs.txt &

In [17]:
public_url = ngrok.connect(8501)

print(public_url)

NgrokTunnel: "https://doorknob-confound-importer.ngrok-free.dev" -> "http://localhost:8501"
